In [1]:
# imports
from openai import OpenAI
import gradio as gr
import pandas as pd
from dotenv import load_dotenv

In [2]:
import os
load_dotenv(override=True)
os.environ["OPENAI_API_KEY"] = os.getenv('OpenAI_API_KEY')

openai = OpenAI()
MODEL_gpt = "gpt-4o-mini"

In [3]:
Llama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
MODEL = "llama3.2"

In [4]:
system_message = "You are a helpful assistant that answers questions based on provided context if you don't know the answer just say 'I don't know'."

In [5]:
def read_excel(file):
    if file is not None:
        df = pd.read_excel(file.name)
        return df
    return None

In [6]:
def chat(message, history, file):
    # Read file if uploaded
    df = read_excel(file)
    
    # Convert Excel to text if exists
    excel_text = ""
    if df is not None:
        excel_text = df.to_string()
    
    # Prepare messages
    prompt = system_message
    if excel_text:
        prompt += f"\nExcel data:\n{excel_text}"

    messages = [{"role": "system", "content": prompt}] + history + [{"role": "user", "content": message}]
    
    # Call your LLaMA chat model
    response = openai.chat.completions.create(
        model=MODEL_gpt,
        messages=messages,
        temperature=0.7,
        max_tokens=512,
    )
    response_text = response.choices[0].message.content
    return response_text

In [7]:
view = gr.ChatInterface(
    fn=chat,
    type="messages",
    additional_inputs=[gr.File(label="Upload Excel file")],
)

view.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
